# TCGA differential expression — limma on Google Colab
Run the cells **in order** in a standard **Python / CPU** runtime. The notebook runs R as a subprocess; no GPU is used.

Upload only the Python preparation exports: `plan_analyses.csv` plus its comparison folders containing `expression.tsv.gz` and `samples.csv`. Do not upload the original full Xena matrix or Python models.

On your Mac, prepare the archive in Terminal:
```bash
cd /Users/mac/Desktop/PrepPython2/SanofiLikeCancerDetector
zip -r donnees_limma.zip resultats_DE_preparation -x '*.pkl' '*/.DS_Store'
```
The R script is embedded below. Faster matrix reading and vectorized QC retain the statistical design: patient blocks when required, limma-trend, robust TREAT (minimum effect 1 on the existing log scale), BH FDR 0.05. PCA is retained on at most 500 genes for visualization. All exported genes enter the statistical tests.

No speedup is guaranteed. Stage timings identify the bottleneck; a heartbeat only means the process is alive, not that a known percentage is complete. Colab resources and session lifetime are limited. This notebook does not automatically resume interrupted comparisons. Inspect QC and warnings before interpreting exploratory results.


## 1. Install R and binary packages
Installation may take several minutes. These dependencies are installed in Colab, not on your Mac.

In [ ]:
import subprocess, os, time, shutil, zipfile, tempfile, threading, queue
from pathlib import Path
from datetime import datetime
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "r-base", "r-bioc-limma",
                "r-cran-statmod", "r-cran-data.table", "r-cran-r.utils",
                "r-cran-matrixstats"], check=True)
subprocess.run(["Rscript", "-e", 'for (p in c("limma","statmod","data.table","R.utils","matrixStats")) { stopifnot(requireNamespace(p, quietly=TRUE)); cat(p, as.character(packageVersion(p)), "\n") }'], check=True)


## 2. Choose output storage
Google Drive is enabled by default so completed files survive loss of the runtime. Authorize mounting when prompted. Set `SAVE_TO_DRIVE = False` to use temporary storage and download results before disconnecting.

In [ ]:
SAVE_TO_DRIVE = True
from google.colab import files
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/TCGA_Limma/results")
else:
    OUTPUT_BASE = Path("/content/limma_results")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_BASE)


## 3. Import preparation exports
Leave `ARCHIVE_ON_DRIVE` empty to upload the ZIP from your Mac. For a large archive, place it in Drive and enter its full path instead. Extraction always happens on the Colab local disk for faster reading.

In [ ]:
ARCHIVE_ON_DRIVE = ""  # e.g. /content/drive/MyDrive/donnees_limma.zip
if ARCHIVE_ON_DRIVE:
    archive = Path(ARCHIVE_ON_DRIVE)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one ZIP archive.")
    archive = Path(next(iter(uploaded)))
    del uploaded
if not archive.is_file() or not zipfile.is_zipfile(archive):
    raise ValueError("A valid ZIP archive is required.")
EXTRACT_DIR = Path(tempfile.mkdtemp(prefix="limma_input_", dir="/content"))
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if not target.is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError("Unsafe archive path.")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Symbolic links are not accepted in input archives.")
    z.extractall(EXTRACT_DIR)
plans = [p for p in EXTRACT_DIR.rglob("plan_analyses.csv") if "__MACOSX" not in p.parts]
if len(plans) != 1:
    raise ValueError(f"Expected one plan_analyses.csv; found {len(plans)}.")
INPUT_DIR = plans[0].parent
print("Input directory:", INPUT_DIR)


## 4. Write the R script
Expand this cell to inspect the full analysis. The same R script can also run locally if its dependencies are installed.

In [ ]:
R_SOURCE = '# AnalyseLimma.R\n# Input: outputs of PrepareDEComparisons.py, already on the Xena log scale.\n# No raw-count method, voom, extra log transform or StandardScaler is used.\n# TREAT tests an effect exceeding MIN_EFFECT on the INPUT log-expression scale.\n# Results are exploratory until expression QC and confounding are reviewed.\n\n# ---- Editable configuration --------------------------------------------------\nPROJECT_DIR <- "/Users/mac/Desktop/PrepPython2/SanofiLikeCancerDetector"\nINPUT_DIR <- file.path(PROJECT_DIR, "resultats_DE_preparation")\nOUTPUT_BASE <- file.path(PROJECT_DIR, "resultats_limma")\nFDR_THRESHOLD <- 0.05\nMIN_EFFECT <- 1.0\nMIN_PATIENTS_PER_GROUP <- 10L\nMIN_PAIRS_FOR_BLOCK_MODEL <- 3L\nUSE_TREND <- TRUE\nUSE_ROBUST <- TRUE\n\n# Optional terminal use: Rscript AnalyseLimma.R /path/to/input /path/to/output\ncli_args <- commandArgs(trailingOnly = TRUE)\nif (length(cli_args) >= 1L) INPUT_DIR <- cli_args[1L]\nif (length(cli_args) >= 2L) OUTPUT_BASE <- cli_args[2L]\n\n# ---- Dependencies ------------------------------------------------------------\nneeded <- c("limma", "statmod", "data.table", "R.utils", "matrixStats")\nmissing_packages <- needed[!vapply(needed, requireNamespace, logical(1), quietly = TRUE)]\nif (length(missing_packages)) {\n  stop("Missing packages: ", paste(missing_packages, collapse = ", "),\n       ". Install them using the commands in the accompanying guide.")\n}\nif (!is.finite(FDR_THRESHOLD) || FDR_THRESHOLD <= 0 || FDR_THRESHOLD >= 1 ||\n    !is.finite(MIN_EFFECT) || MIN_EFFECT < 0) stop("Invalid statistical thresholds.")\n\nstage <- function(label, expr) {\n  cat(format(Sys.time(), "%H:%M:%S"), "START", label, "\\n"); flush.console()\n  start <- proc.time()[3L]\n  value <- force(expr)\n  cat(format(Sys.time(), "%H:%M:%S"), "DONE", label,\n      sprintf("(%.1f s)", proc.time()[3L] - start), "\\n"); flush.console()\n  value\n}\n\nwrite_table <- function(x, path) {\n  write.csv(x, path, row.names = FALSE, na = "", fileEncoding = "UTF-8")\n}\n\nread_metadata <- function(path) {\n  # Explicit character columns preserve patient IDs and labels.\n  read.csv(path, stringsAsFactors = FALSE, check.names = FALSE, colClasses = "character")\n}\n\nsave_pdf <- function(path, draw, width = 9, height = 7) {\n  grDevices::pdf(path, width = width, height = height)\n  on.exit(grDevices::dev.off(), add = TRUE)\n  draw()\n}\n\n# Use precisely the paths recorded by the Python preparation manifest.\nresolve_input <- function(relative_path) {\n  if (is.na(relative_path) || !nzchar(relative_path) || grepl("^(/|[A-Za-z]:)", relative_path) ||\n      ".." %in% strsplit(gsub("\\\\\\\\", "/", relative_path), "/", fixed = TRUE)[[1L]]) {\n    stop("Invalid relative path in plan_analyses.csv: ", relative_path)\n  }\n  full <- file.path(INPUT_DIR, relative_path)\n  if (!file.exists(full)) stop("Missing input file: ", full)\n  full\n}\n\nmake_qc <- function(y, samples, out) {\n  colors <- ifelse(samples$group == "Normal", "#2474B5", "#C84043")\n  qc <- data.frame(\n    sample = samples$sample, patient_id = samples$patient_id, group = samples$group,\n    mean_log_expression = colMeans(y), median_log_expression = matrixStats::colMedians(y),\n    sd_log_expression = matrixStats::colSds(y), stringsAsFactors = FALSE\n  )\n  write_table(qc, file.path(out, "qc_echantillons.csv"))\n  save_pdf(file.path(out, "qc_distributions.pdf"), function() {\n    # No sample is automatically deleted based on these distributions.\n    boxplot(y, outline = FALSE, names = FALSE, col = colors, border = colors,\n            main = "Sample expression distributions", xlab = "Samples (metadata order)",\n            ylab = "Input log2(norm_value + 1)")\n    legend("topright", c("Normal", "Tumor"), fill = c("#2474B5", "#C84043"), bty = "n")\n  })\n  variances <- matrixStats::rowVars(y)\n  selected <- head(order(variances, decreasing = TRUE), min(500L, nrow(y)))\n  pca <- stage("PCA: 500 genes maximum, visualization only", prcomp(t(y[selected, , drop = FALSE]), center = TRUE, scale. = FALSE, rank. = 2L))\n  if (ncol(pca$x) < 2L) stop("Insufficient expression rank for a two-dimensional PCA.")\n  explained <- 100 * pca$sdev^2 / sum(pca$sdev^2)\n  coordinates <- data.frame(sample = samples$sample, patient_id = samples$patient_id,\n                            group = samples$group, PC1 = pca$x[, 1L], PC2 = pca$x[, 2L])\n  write_table(coordinates, file.path(out, "qc_PCA_coordonnees.csv"))\n  save_pdf(file.path(out, "qc_PCA.pdf"), function() {\n    plot(coordinates$PC1, coordinates$PC2, col = colors, pch = 19,\n         xlab = sprintf("PC1 (%.1f%%)", explained[1L]),\n         ylab = sprintf("PC2 (%.1f%%)", explained[2L]),\n         main = sprintf("PCA: %d most variable genes (visualization only)", length(selected)))\n    legend("topright", c("Normal", "Tumor"), col = c("#2474B5", "#C84043"), pch = 19, bty = "n")\n  })\n}\n\nanalyse_comparison <- function(entry, run_dir) {\n  name <- entry$comparison_id\n  if (is.na(name) || !grepl("^[A-Za-z0-9_-]+$", name)) stop("Invalid comparison name.")\n  out <- file.path(run_dir, name)\n  dir.create(out, recursive = TRUE, showWarnings = FALSE)\n  samples <- read_metadata(resolve_input(entry$samples_file))\n  required <- c("sample", "patient_id", "group", "gdc_project", "_primary_disease")\n  if (!all(required %in% names(samples))) stop("Missing required metadata columns.")\n  if (anyNA(samples[, required]) || any(samples$sample == "") || any(samples$patient_id == "")) {\n    stop("Missing sample, patient or group metadata.")\n  }\n  if (anyDuplicated(samples$sample)) stop("Duplicate sample identifiers.")\n  if (!all(samples$group %in% c("Normal", "Tumor"))) stop("Unexpected group labels.")\n  if (anyDuplicated(samples[, c("patient_id", "group")])) {\n    stop("Repeated observations for the same patient and condition require review.")\n  }\n  if (!all(samples$gdc_project == entry$project) ||\n      !all(samples[["_primary_disease"]] == entry$disease)) stop("Project/disease mismatch with the plan.")\n  table <- stage("Read expression matrix", data.table::fread(\n    resolve_input(entry$expression_file), data.table = FALSE, check.names = FALSE,\n    colClasses = list(character = 1L)))\n  gene_ids <- table[[1L]]\n  if (anyNA(gene_ids) || anyDuplicated(gene_ids)) stop("Invalid gene identifiers.")\n  table <- table[, -1L, drop = FALSE]\n  rownames(table) <- gene_ids\n  if (!all(vapply(table, is.numeric, logical(1)))) stop("Expression contains non-numeric columns.")\n  y <- as.matrix(table)\n  rm(table)\n  if (anyDuplicated(rownames(y)) || anyDuplicated(colnames(y))) stop("Duplicate expression identifiers.")\n  if (!setequal(colnames(y), samples$sample)) stop("Expression and metadata sample sets differ.")\n  # Explicit alignment: never assume the two files are ordered identically.\n  samples <- samples[match(colnames(y), samples$sample), , drop = FALSE]\n  stopifnot(identical(colnames(y), samples$sample))\n  if (nrow(y) < 10L || !all(is.finite(y))) stop("Too few genes or nonfinite expression values.")\n  if (any(matrixStats::rowVars(y) == 0)) stop("Constant genes remain; inspect the preparation output.")\n  samples$group <- factor(samples$group, levels = c("Normal", "Tumor"))\n  n_normal <- sum(samples$group == "Normal")\n  n_tumor <- sum(samples$group == "Tumor")\n  if (n_normal < MIN_PATIENTS_PER_GROUP || n_tumor < MIN_PATIENTS_PER_GROUP) {\n    stop("Fewer than the configured minimum independent patients per group.")\n  }\n  if (n_normal != as.integer(entry$n_normal) || n_tumor != as.integer(entry$n_tumor) ||\n      nrow(y) != as.integer(entry$n_genes_exported)) stop("File dimensions disagree with plan_analyses.csv.")\n  normal_ids <- samples$patient_id[samples$group == "Normal"]\n  tumor_ids <- samples$patient_id[samples$group == "Tumor"]\n  n_pairs <- length(intersect(normal_ids, tumor_ids))\n  write_table(samples, file.path(out, "echantillons_utilises.csv"))\n  stage("QC distributions and PCA", make_qc(y, samples, out))\n\n  # Mean model: one coefficient per condition. Contrast is Tumor minus Normal.\n  design <- model.matrix(~ 0 + group, data = samples)\n  colnames(design) <- c("Normal", "Tumor")\n  rownames(design) <- samples$sample\n  if (qr(design)$rank != ncol(design)) stop("Rank-deficient design matrix.")\n  write_table(data.frame(sample = rownames(design), design, check.names = FALSE),\n              file.path(out, "matrice_design.csv"))\n  correlation <- NA_real_\n  method <- "independent_patients"\n  if (n_pairs > 0L) {\n    # Guard against unstable block-correlation estimation from very few pairs.\n    # This is a workflow choice, not a universal mathematical threshold.\n    if (n_pairs < MIN_PAIRS_FOR_BLOCK_MODEL) {\n      stop(sprintf("Only %d paired patients: block-model design needs review. QC files were saved; no independence assumption was substituted.", n_pairs))\n    }\n    block <- factor(samples$patient_id)\n    dc <- stage("duplicateCorrelation: all genes, patient blocks (may be slow)", limma::duplicateCorrelation(y, design = design, block = block))\n    correlation <- dc$consensus.correlation\n    if (!is.finite(correlation) || abs(correlation) >= 1) stop("Invalid estimated within-patient correlation.")\n    fit <- stage("lmFit with patient blocks", limma::lmFit(y, design, block = block, correlation = correlation))\n    method <- "patient_block_duplicateCorrelation"\n  } else {\n    fit <- stage("lmFit independent patients", limma::lmFit(y, design))\n  }\n  contrast <- limma::makeContrasts(Tumor - Normal, levels = design)\n  fit <- limma::contrasts.fit(fit, contrast)\n  if (any(fit$df.residual <= 0)) stop("Insufficient residual degrees of freedom.")\n\n  # Empirical Bayes TREAT: H0 = |effect| <= MIN_EFFECT.\n  # For this preprocessed log2(norm_value + 1) matrix, the effect is a difference\n  # on that scale, NOT necessarily log2 of a raw-count expression ratio.\n  fit <- stage("TREAT statistical tests", limma::treat(fit, lfc = MIN_EFFECT, trend = USE_TREND, robust = USE_ROBUST))\n  result <- limma::topTreat(fit, coef = 1L, number = Inf, adjust.method = "BH", sort.by = "none")\n  idx <- match(rownames(result), rownames(y))\n  result <- data.frame(\n    gene = rownames(result), comparison_id = name, cancer = entry$disease,\n    project = entry$project, tumor_sites = entry$tumor_sites, normal_sites = entry$normal_sites,\n    n_normal = n_normal, n_tumor = n_tumor, n_paired_patients = n_pairs,\n    mean_normal_log_expression = rowMeans(y[, samples$group == "Normal", drop = FALSE])[idx],\n    mean_tumor_log_expression = rowMeans(y[, samples$group == "Tumor", drop = FALSE])[idx],\n    effect_log2_scale = result$logFC, average_log_expression = result$AveExpr,\n    moderated_t_treat = result$t, p_value_treat = result$P.Value, fdr_bh_treat = result$adj.P.Val,\n    stringsAsFactors = FALSE, row.names = NULL\n  )\n  if (any(!is.finite(result$p_value_treat)) || any(!is.finite(result$fdr_bh_treat))) {\n    stop("Nonfinite test results: inspect model fit.")\n  }\n  result$selected <- result$fdr_bh_treat <= FDR_THRESHOLD\n  result$direction <- ifelse(result$selected,\n                              ifelse(result$effect_log2_scale > 0, "Up_in_tumor", "Down_in_tumor"),\n                              "Not_selected")\n  result$test <- "limma_TREAT"\n  result$minimum_effect_tested <- MIN_EFFECT\n  result$fdr_threshold <- FDR_THRESHOLD\n  result <- result[order(result$fdr_bh_treat, -abs(result$effect_log2_scale)), , drop = FALSE]\n  write_table(result, file.path(out, "resultats_complets.csv"))\n  write_table(result[result$selected, , drop = FALSE], file.path(out, "genes_retenus.csv"))\n  write_table(result[result$direction == "Up_in_tumor", , drop = FALSE], file.path(out, "genes_surexprimes.csv"))\n  write_table(result[result$direction == "Down_in_tumor", , drop = FALSE], file.path(out, "genes_sous_exprimes.csv"))\n\n  cat("CSV results saved; writing plots.\\n"); flush.console()\n  colors <- ifelse(result$direction == "Up_in_tumor", "#C84043",\n                   ifelse(result$direction == "Down_in_tumor", "#2474B5", "#999999"))\n  save_pdf(file.path(out, "volcano.pdf"), function() {\n    plot(result$effect_log2_scale, -log10(pmax(result$fdr_bh_treat, .Machine$double.xmin)),\n         pch = 16, cex = 0.45, col = colors, main = name,\n         xlab = "Tumor - Normal effect on input log2 scale", ylab = "-log10(BH-adjusted TREAT p-value)")\n    abline(v = c(-MIN_EFFECT, MIN_EFFECT), h = -log10(FDR_THRESHOLD), lty = 2, col = "#555555")\n  })\n  save_pdf(file.path(out, "MA_plot.pdf"), function() {\n    plot(result$average_log_expression, result$effect_log2_scale,\n         pch = 16, cex = 0.45, col = colors, main = name,\n         xlab = "Mean input log expression", ylab = "Tumor - Normal effect")\n    abline(h = c(-MIN_EFFECT, 0, MIN_EFFECT), lty = c(2, 1, 2), col = "#555555")\n  })\n  save_pdf(file.path(out, "qc_moyenne_variance.pdf"), function() {\n    limma::plotSA(fit, main = paste(name, "- mean/variance trend"))\n  })\n  capture.output(list(\n    comparison = as.list(entry), contrast = "Tumor - Normal", method = method,\n    n_paired_patients = n_pairs, consensus_correlation = correlation,\n    use_trend = USE_TREND, use_robust = USE_ROBUST,\n    test = "TREAT: absolute effect exceeds MIN_EFFECT", minimum_effect = MIN_EFFECT,\n    fdr_BH_within_comparison = FDR_THRESHOLD,\n    scale = "Existing log2(norm_value + 1), no additional transformation",\n    limitations = c("No additional age/sex/batch/purity covariates fitted in this version.",\n      "Upstream EB++ processing is inherited; downstream modelling cannot undo upstream confounding.",\n      "Inspect QC and tissue details before interpreting these exploratory results.",\n      "BH correction is within each comparison, not across all cancers.",\n      "Normal samples are patient-derived controls, not automatically healthy-donor tissue.",\n      "Association does not establish causality or a therapeutic target.")\n  ), file = file.path(out, "parametres_et_limites.txt"))\n  data.frame(comparison_id = name, status = "completed_exploratory", n_normal = n_normal,\n    n_tumor = n_tumor, n_paired_patients = n_pairs, n_genes_tested = nrow(result),\n    n_up = sum(result$direction == "Up_in_tumor"), n_down = sum(result$direction == "Down_in_tumor"),\n    correlation = correlation, model = method, message = "", stringsAsFactors = FALSE)\n}\n\nrun_limma <- function() {\n  start <- Sys.time()\n  plan_path <- file.path(INPUT_DIR, "plan_analyses.csv")\n  if (!file.exists(plan_path)) stop("Missing plan: ", plan_path, ". Check INPUT_DIR.")\n  plan <- read_metadata(plan_path)\n  required <- c("comparison_id", "status", "project", "disease", "tumor_sites", "normal_sites",\n                "expression_file", "samples_file", "n_normal", "n_tumor", "n_genes_exported")\n  if (!all(required %in% names(plan))) stop("Unexpected preparation manifest schema.")\n  if (anyDuplicated(plan$comparison_id)) stop("Duplicate comparison IDs in manifest.")\n  work <- plan[plan$status == "exported_for_expression_qc", , drop = FALSE]\n  if (!nrow(work)) stop("No exported comparisons found. Inspect Python plan_analyses.csv.")\n  # Never overwrite an earlier run or mix old and new results.\n  run_dir <- file.path(OUTPUT_BASE, paste0("run_", format(start, "%Y%m%d_%H%M%S")))\n  if (dir.exists(run_dir)) stop("Run directory already exists: ", run_dir)\n  dir.create(run_dir, recursive = TRUE, showWarnings = FALSE)\n  write_table(plan, file.path(run_dir, "plan_entree.csv"))\n  writeLines(c(paste("Start:", start), paste("Input:", normalizePath(INPUT_DIR)),\n    paste("FDR:", FDR_THRESHOLD), paste("Minimum effect:", MIN_EFFECT),\n    paste("Trend:", USE_TREND), paste("Robust:", USE_ROBUST)), file.path(run_dir, "execution.txt"))\n  on.exit(capture.output(sessionInfo(), file = file.path(run_dir, "sessionInfo.txt")), add = TRUE)\n  summaries <- vector("list", nrow(work))\n  for (i in seq_len(nrow(work))) {\n    entry <- work[i, , drop = FALSE]\n    cat(sprintf("\\n[%d/%d] %s\\n", i, nrow(work), entry$comparison_id))\n    warnings_seen <- character()\n    summaries[[i]] <- tryCatch(\n      withCallingHandlers(analyse_comparison(entry, run_dir), warning = function(w) {\n        warnings_seen <<- c(warnings_seen, conditionMessage(w))\n        invokeRestart("muffleWarning")\n      }),\n      error = function(e) {\n        cat("Not completed:", conditionMessage(e), "\\n")\n        data.frame(comparison_id = entry$comparison_id, status = "error_or_design_review_required",\n          n_normal = NA_integer_, n_tumor = NA_integer_, n_paired_patients = NA_integer_,\n          n_genes_tested = NA_integer_, n_up = NA_integer_, n_down = NA_integer_,\n          correlation = NA_real_, model = "", message = conditionMessage(e), stringsAsFactors = FALSE)\n      }\n    )\n    if (length(warnings_seen)) {\n      writeLines(unique(warnings_seen), file.path(run_dir, paste0(entry$comparison_id, "_warnings.txt")))\n      cat("Warnings recorded; review", entry$comparison_id, "_warnings.txt\\n")\n      summaries[[i]]$message <- paste(c(summaries[[i]]$message, unique(warnings_seen)), collapse = " | ")\n    }\n    write_table(do.call(rbind, summaries[seq_len(i)]), file.path(run_dir, "synthese_analyses.csv"))\n    if (summaries[[i]]$status == "completed_exploratory") {\n      cat(sprintf("Completed: %d up, %d down, %d genes tested.\\n",\n          summaries[[i]]$n_up, summaries[[i]]$n_down, summaries[[i]]$n_genes_tested))\n    }\n    invisible(gc())\n  }\n  final <- do.call(rbind, summaries)\n  elapsed <- as.numeric(difftime(Sys.time(), start, units = "mins"))\n  cat(sprintf("\\nFinished in %.1f min. Completed: %d/%d.\\nResults: %s\\n", elapsed,\n              sum(final$status == "completed_exploratory"), nrow(final), run_dir))\n  cat("Inspect synthese_analyses.csv, warnings and QC before interpretation.\\n")\n  if (any(final$status != "completed_exploratory")) stop("Some comparisons failed: inspect synthese_analyses.csv; successful outputs are preserved.")\n  invisible(final)\n}\n\nrun_limma()\n'
R_SCRIPT = Path("/content/AnalyseLimmaCloud.R")
R_SCRIPT.write_text(R_SOURCE, encoding="utf-8")
print("R script ready:", R_SCRIPT)


## 5. Run all exported comparisons
The log shows the active step and its duration. In particular, `duplicateCorrelation` may take time when accounting for samples from the same patient. Do not remove patient blocking just to reduce runtime. The execution log and R script are saved with results. A new execution creates a new output session; previous results are preserved.

In [ ]:
SESSION_DIR = OUTPUT_BASE / ("session_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
SESSION_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(R_SCRIPT, SESSION_DIR / R_SCRIPT.name)
log_path = SESSION_DIR / "console.log"
environment = os.environ.copy()
# Avoid oversubscribing CPU threads; comparisons remain sequential to limit RAM.
environment.update(OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")
proc = subprocess.Popen(["Rscript", str(R_SCRIPT), str(INPUT_DIR), str(SESSION_DIR)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=environment)
messages = queue.Queue()
def read_output():
    for line in proc.stdout:
        messages.put(line)
    messages.put(None)
threading.Thread(target=read_output, daemon=True).start()
started = time.monotonic()
try:
    with log_path.open("w", encoding="utf-8", buffering=1) as log:
        while True:
            try:
                line = messages.get(timeout=30)
            except queue.Empty:
                print(f"Process still running — elapsed {(time.monotonic()-started)/60:.1f} min", flush=True)
                continue
            if line is None:
                break
            print(line, end="", flush=True)
            log.write(line)
    return_code = proc.wait()
except BaseException:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
    raise
print("Saved outputs:", SESSION_DIR)
if return_code:
    raise RuntimeError("R reported an error. Read console.log and synthese_analyses.csv. You can still run the download cell to recover partial results.")


## 6. Download results
Run this cell even after an analysis error to recover completed outputs and logs. The ZIP contains all files from this execution. If Drive is enabled, an extra ZIP copy is saved there. Review `synthese_analyses.csv`, per-comparison QC and warning files: a created folder alone does not mean its analysis succeeded.

In [ ]:
if "SESSION_DIR" not in globals() or not SESSION_DIR.is_dir():
    raise RuntimeError("Run the analysis cell first.")
zip_base = Path("/content") / SESSION_DIR.name
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=SESSION_DIR.parent, base_dir=SESSION_DIR.name))
if SAVE_TO_DRIVE:
    shutil.copy2(zip_path, OUTPUT_BASE / zip_path.name)
print("Download:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(zip_path))


## Interpretation and reproducibility
- The input is already on the Xena `log2(norm_value + 1)` scale: no DESeq2, voom, re-log transformation or model standardization is applied.
- The effect is a tumor-minus-normal difference on that input scale, not necessarily a raw-count log fold change.
- TREAT tests whether the absolute effect exceeds 1; BH adjustment is within each comparison.
- No age, sex, purity or additional batch covariates are added in this version. Tissue and expression QC still require review.
- `sessionInfo.txt` records installed versions. Package versions can differ from the Mac run.
- Faster I/O and vectorized QC are implemented; R/Colab execution has not been validated on your actual data in this environment.
- Colab documentation: https://research.google.com/colaboratory/faq.html
